# Train CodeBERT for stage 1 and test it on domains it has never seen

**What this notebook does, in one paragraph.** Stage 1 of C4 ranks code files for every requirement. This notebook fine-tunes CodeBERT so that a requirement and the code that implements it end up close together in vector space, then measures how well the trained model finds the right code in **new domains it was not trained on**. It also compares it with TF-IDF and MiniLM, draws the graphs, runs a control test, and saves the model.

**The data.** `data/synthetic_clean`: requirement / code pairs written by an LLM, cleaned by rules (see section 3). Each item = one requirement, the code that implements it, and 2 *hard negatives* (code that looks similar but does something else). Items belong to a *domain* (a group, for example "library loans"). The split is by domain, never by item, so the test domains are really new.

**How to read the result.** Training and test data come from the same writers, so a good score proves that CodeBERT *can learn* this kind of data. It does **not** prove that it works on real projects. That is what the cross-validation notebook (`codebert_cross_validation.ipynb`) and the real-project test notebook (`codebert_stage1_test.ipynb`) are for.

**Run it:** top to bottom. On a Mac it takes a long time (use `QUICK = True` in section 2 to check that everything works in a few minutes); on a Colab T4 GPU the full run is about 10 to 20 minutes.

## 1. Environment

This cell works on your Mac and in Google Colab. On Colab it installs what is missing and asks you to upload `colab_bundle.zip` (the small project files and datasets this notebook needs). On your Mac it just finds the project folder. It also picks the fastest device it can find: a GPU (`cuda` on Colab, `mps` on a Mac) or the CPU.

In [ ]:
import os, sys, json, random, time, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers', 'scikit-learn', 'pydantic', 'matplotlib'], check=False)
    if not Path('/content/c4/agents').exists():
        from google.colab import files
        uploaded = files.upload()                       # choose colab_bundle.zip
        import zipfile; zipfile.ZipFile(next(iter(uploaded))).extractall('/content/c4')
    ROOT = Path('/content/c4')
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / 'agents').is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent                              # find the project folder from wherever the notebook runs
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np, torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('project folder:', ROOT); print('device:', DEVICE, '| on Colab:', IN_COLAB)

## Plot style

One small cell that makes every graph below look the same (clean grid, no top/right border) and gives each method its own colour, so a method has the same colour in every graph.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
COLORS = {'TF-IDF': '#9aa5b1', 'MiniLM': '#4c78a8', 'CodeBERT (not trained)': '#c9a227', 'CodeBERT (fine-tuned)': '#d1495b',
          'TF-IDF + MiniLM': '#2f6f4e', 'TF-IDF + CodeBERT (fine-tuned)': '#9b5de5', 'TF-IDF + MiniLM + CodeBERT': '#e07a1f'}
def color(name):
    return COLORS.get(name, '#444444')

def show_table(rows, columns, width=44):
    """Print {name: {metric: value}} as an aligned text table."""
    print(f"{'':{width}s}" + ''.join(f'{c:>10s}' for c in columns))
    for name, r in rows.items():
        print(f'{name:{width}s}' + ''.join(f'{r[c]:10.3f}' for c in columns))

## 2. Settings

Everything you may want to change is here.

* `BASE`: the model that is fine-tuned. It is downloaded from Hugging Face the first time (about 500 MB).
* `EPOCHS`: how many times the training data is seen. `BATCH`: requirements per step (each brings 3 codes, so memory use is 3x). If you get *CUDA out of memory*, use 8.
* `TAU`: the temperature of the loss. Small values make the model separate right from wrong code harder.
* `QUICK`: a tiny run to check that the notebook works (64 items, 1 epoch).

In [ ]:
BASE = 'microsoft/codebert-base'
EPOCHS, BATCH, LR, TAU = 3, 16, 2e-5, 0.05
TEST_GROUPS, SPLIT_SEED, TRAIN_SEED = 6, 33, 13     # how many domains are held out for the test, and the random seeds
QUICK = False                                        # True = tiny run, only to see that everything works
OUT = (Path('/content/out') if IN_COLAB else ROOT / 'models' / 'c4-codebert-new')   # new folder: your current model is not overwritten
if QUICK:
    EPOCHS, BATCH = 1, 8
print('model:', BASE, '| epochs:', EPOCHS, '| batch:', BATCH, '| output folder:', OUT)

## 3. The data and how it was cleaned

The raw pairs were cleaned before training. Each item was dropped as a whole if any rule failed:
1. the code, with its `package` / `import` / licence header removed, must still contain text and have balanced braces;
2. no label-revealing word (for example *wrong*, *alternative*, *decoy*) may appear in a class name or requirement, otherwise the model could learn the word instead of the meaning;
3. every hard negative must differ from the right code;
4. duplicate requirements and duplicate code are kept once.

The report written by the cleaning step shows what was kept.

In [ ]:
report = json.loads((ROOT / 'data' / 'synthetic_clean' / 'clean_report.json').read_text())
print('items read:', report['items_read'], '| kept:', report['items_kept'], '| domains (groups):', report['groups'])
print('dropped:', {k: v for k, v in report.items() if k not in ('items_read', 'items_kept', 'groups', 'valid_groups', 'rows')})
print('rows written:', report['rows'])

### Load the pairs and look at one

`load_items` turns the file into one record per requirement: its text, its right code (`pos`) and its two hard negatives (`hard`).

In [ ]:
DATA = ROOT / 'data' / 'synthetic_clean'

def load_items(path):
    """One item per requirement: {group, id, query, pos, hard[2]}."""
    items = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        item = items.setdefault(r['source_id'], {'group': r['source_id'].split('#')[0], 'id': r['source_id'], 'query': r['query'], 'pos': None, 'hard': []})
        if r['kind'] == 'pos': item['pos'] = r['doc']
        else: item['hard'].append(r['doc'])
    return [i for i in items.values() if i['pos'] and len(i['hard']) == 2]

train_all, valid_items = load_items(DATA / 'train.jsonl'), load_items(DATA / 'valid.jsonl')
ex = train_all[0]
print('REQUIREMENT:', ex['query'], '\n\nRIGHT CODE:\n', ex['pos'][:350], '...\n\nHARD NEGATIVE 1 (looks similar, does something else):\n', ex['hard'][0][:350], '...')

### Split by domain

Test domains are taken out of the training pool with a fixed seed. **Training** learns from `train`; **validation** (the domains the cleaner already set aside) only chooses the best epoch; **test** is touched once at the end. The bar chart shows how many items each domain has and which split it belongs to.

In [ ]:
groups = sorted({i['group'] for i in train_all})
test_groups = set(random.Random(SPLIT_SEED).sample(groups, TEST_GROUPS))
data = {'train': [i for i in train_all if i['group'] not in test_groups], 'valid': valid_items, 'test': [i for i in train_all if i['group'] in test_groups]}
if QUICK:
    data['train'] = random.Random(TRAIN_SEED).sample(data['train'], 64)
print({k: f"{len(v)} items, {len({i['group'] for i in v})} domains" for k, v in data.items()})

counts = {}
for split, its in data.items():
    for it in its: counts.setdefault((split, it['group']), 0); counts[(split, it['group'])] += 1
fig, ax = plt.subplots(figsize=(11, 3.2))
shown = sorted(counts, key=lambda k: ('train', 'valid', 'test').index(k[0]))
ax.bar(range(len(shown)), [counts[k] for k in shown], color=[{'train': '#4c78a8', 'valid': '#e0a030', 'test': '#d1495b'}[k[0]] for k in shown])
ax.set_xticks([]); ax.set_xlabel('domains (each bar = one domain)'); ax.set_ylabel('items')
ax.set_title('Items per domain: blue = train, orange = validation, red = test (never seen in training)')
plt.show()

## 4. How a method is scored

For a requirement we rank code files and ask: *where does the right code end up?*

* **Group pool**: the candidates are all codes of the requirement's own domain (the right codes, the hard negatives and the other requirements' codes). This is the demanding test, because all candidates look alike.
* **Mixed pool**: the candidates of all test domains together.
* **R@k** (recall at k): the share of requirements whose right code is among the k best. **MRR**: the average of 1 / rank of the right code (1.0 = always first).

Three scorers are defined here: **TF-IDF** (word overlap), **MiniLM** (a general sentence model) and **CodeBERT** used as a *bi-encoder*: the requirement and the code are turned into vectors separately (mean pooling of the token vectors) and compared by cosine similarity. `fused` adds the z-scores of two scorers, exactly as stage 1 fuses its signals.

In [ ]:
import re

_STOP = frozenset("public private protected static final void class interface enum import package return new this null true false int long double float char byte short boolean string extends implements throws throw try catch finally if else for while do switch case break continue default abstract synchronized instanceof super override".split())
def split_identifiers(text):
    """addPatientRecord -> 'add patient record' (the same preparation stage 1 gives code)."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text); text = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", text)
    toks = re.findall(r"[A-Za-z]+", text.replace("_", " "))
    return " ".join(t for t in (t.lower() for t in toks) if len(t) > 1 and t not in _STOP)

KS = (1, 3, 5, 10)
def pool_of(items):
    docs, right = [], []
    for it in items:
        right.append(len(docs)); docs += [it['pos'], *it['hard']]
    return docs, right

def zscore(m): return (m - m.mean(axis=1, keepdims=True)) / (m.std(axis=1, keepdims=True) + 1e-9)
def rank_of_right(scores, right): return np.array([(scores[i] > scores[i, r]).sum() + 1 for i, r in enumerate(right)])
def metrics_from_ranks(ranks):
    out = {f'R@{k}': round(float((ranks <= k).mean()), 3) for k in KS}; out['MRR'] = round(float((1 / ranks).mean()), 3); return out

def evaluate(score_fn, items, keep_ranks=False):
    """Group pool (average over domains) and mixed pool for one scorer (queries, codes) -> score matrix."""
    by_group = {}
    for it in items: by_group.setdefault(it['group'], []).append(it)
    per_group = {}
    for g, its in by_group.items():
        docs, right = pool_of(its); per_group[g] = rank_of_right(score_fn([i['query'] for i in its], docs), right)
    docs, right = pool_of(items); mixed = rank_of_right(score_fn([i['query'] for i in items], docs), right)
    out = {'group pool': {k: round(float(np.mean([metrics_from_ranks(r)[k] for r in per_group.values()])), 3) for k in metrics_from_ranks(mixed)},
           'mixed pool': metrics_from_ranks(mixed)}
    if keep_ranks: out['_ranks'] = {'mixed': mixed, 'per_group': per_group}
    return out

def fused(a, b): return lambda q, c: zscore(a(q, c)) + zscore(b(q, c))

def tfidf_scorer():
    from sklearn.feature_extraction.text import TfidfVectorizer
    def score(queries, codes):
        vec = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit(queries + [split_identifiers(c) for c in codes])
        return (vec.transform(queries) @ vec.transform([split_identifiers(c) for c in codes]).T).toarray()
    return score

def encoder_scorer(encode_q, encode_c):
    cache = {}
    def score(queries, codes):
        prepared = [split_identifiers(c) for c in codes]
        missing = list(dict.fromkeys(c for c in prepared if c not in cache))      # every code is embedded only once
        if missing:
            for c, v in zip(missing, encode_c(missing)): cache[c] = v
        return encode_q(queries) @ np.stack([cache[c] for c in prepared]).T
    return score

def minilm_scorer():
    from sentence_transformers import SentenceTransformer
    local = ROOT / 'models' / 'all-MiniLM-L6-v2'
    model = SentenceTransformer(str(local) if local.is_dir() else 'all-MiniLM-L6-v2')
    enc = lambda texts: model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return encoder_scorer(enc, enc)

### CodeBERT as a bi-encoder

`MeanPoolEncoder` wraps the Hugging Face model. `encode_tensor` keeps gradients (needed for training); `encode` is for testing. Requirements are cut at 64 tokens, code at 256 tokens, which is what the model reads.

In [ ]:
from transformers import AutoModel, AutoTokenizer

class MeanPoolEncoder:
    """CodeBERT as a bi-encoder: mean of the token vectors, length 1."""
    def __init__(self, name, device):
        self.tok, self.model, self.device = AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).to(device), device
    def encode_tensor(self, texts, max_len):
        batch = self.tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(self.device)
        hidden = self.model(**batch).last_hidden_state; mask = batch['attention_mask'].unsqueeze(-1).to(hidden.dtype)
        return torch.nn.functional.normalize((hidden * mask).sum(1) / mask.sum(1).clamp(min=1), dim=-1)
    def encode(self, texts, max_len, batch=32):
        self.model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(texts), batch): out.append(self.encode_tensor(texts[i:i + batch], max_len).cpu().numpy())
        return np.concatenate(out)

def codebert_scorer(enc):
    return encoder_scorer(lambda q: enc.encode(q, 64), lambda c: enc.encode(c, 256))

## 5. Before training: the baselines on the test domains

These numbers are the bar to beat. **CodeBERT (not trained)** is the original model used the same way. It is expected to be poor: CodeBERT was pre-trained to predict masked tokens, not to compare sentences, so its raw vectors are not useful for matching. That is exactly why fine-tuning is needed.

In [ ]:
t0 = time.time()
tf, mini = tfidf_scorer(), minilm_scorer()
enc = MeanPoolEncoder(BASE, DEVICE)
raw = codebert_scorer(enc)
results = {'TF-IDF': evaluate(tf, data['test']), 'MiniLM': evaluate(mini, data['test']), 'TF-IDF + MiniLM': evaluate(fused(tf, mini), data['test']),
           'CodeBERT (not trained)': evaluate(raw, data['test'], keep_ranks=True)}
print(f'baselines took {time.time() - t0:.0f} s\n')
for pool in ('group pool', 'mixed pool'):
    print(pool); show_table({k: v[pool] for k, v in results.items()}, ['R@1', 'R@3', 'R@5', 'R@10', 'MRR']); print()

## 6. Training

The loss is **contrastive**. In every step a batch of requirements is taken. Each requirement must score its *own* code higher than (a) its 2 hard negatives and (b) the codes of all other requirements in the batch. Cross-entropy over those scores pulls the right pair together and pushes the wrong ones apart. The learning rate warms up for the first 10% of the steps and then falls linearly to 0.

After every epoch the model is scored on the validation domains and the **best epoch is kept**. The test domains are never used here.

In [ ]:
def train(enc, items, valid, epochs, batch, lr, tau):
    """Fine-tune the encoder. Returns {'step_loss': [...], 'epoch_valid_R@1': [...]}; the best epoch is kept."""
    random.seed(TRAIN_SEED); torch.manual_seed(TRAIN_SEED)
    opt = torch.optim.AdamW(enc.model.parameters(), lr=lr, weight_decay=0.01)
    steps = max(1, epochs * (len(items) // batch))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, steps // 10)) * max(0.0, 1 - s / steps))
    history, best, best_state = {'step_loss': [], 'epoch_valid_R@1': []}, -1.0, None
    for epoch in range(1, epochs + 1):
        enc.model.train(); order = random.sample(items, len(items)); started = time.time()
        for b in range(0, len(order) - batch + 1, batch):
            chunk = order[b:b + batch]
            q = enc.encode_tensor([i['query'] for i in chunk], 64)
            pos = enc.encode_tensor([split_identifiers(i['pos']) for i in chunk], 256)
            hard = enc.encode_tensor([split_identifiers(h) for i in chunk for h in i['hard']], 256)
            loss = torch.nn.functional.cross_entropy(q @ torch.cat([pos, hard]).T / tau, torch.arange(len(chunk), device=q.device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(enc.model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
            history['step_loss'].append(float(loss))
            if (b // batch) % 20 == 0: print(f'epoch {epoch}  step {b // batch}/{len(order) // batch}  loss {float(loss):.3f}  ({time.time() - started:.0f} s)')
        score = evaluate(codebert_scorer(enc), valid)['group pool']['R@1']; history['epoch_valid_R@1'].append(score)
        print(f'--> epoch {epoch} finished: validation R@1 = {score:.3f}')
        if score > best: best, best_state = score, {k: v.detach().clone() for k, v in enc.model.state_dict().items()}
    enc.model.load_state_dict(best_state); print('kept the epoch with validation R@1', round(best, 3))
    return history

history = train(enc, data['train'], data['valid'], EPOCHS, BATCH, LR, TAU)

### Graph 1: did the model learn?

*Left:* the training loss, step by step (grey) and smoothed (red). It should fall. *Right:* the validation R@1 after each epoch; the best epoch is kept. If the loss falls but validation does not rise, the model is only memorising.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
loss = np.array(history['step_loss']); win = max(1, len(loss) // 20)
a1.plot(loss, color='#cccccc', label='each step'); a1.plot(np.convolve(loss, np.ones(win) / win, mode='valid') if len(loss) > win else loss, color='#d1495b', label='smoothed')
a1.set_xlabel('training step'); a1.set_ylabel('loss'); a1.set_title('Training loss'); a1.legend()
v = history['epoch_valid_R@1']; a2.plot(range(1, len(v) + 1), v, marker='o', color='#2f6f4e'); a2.set_xticks(range(1, len(v) + 1))
a2.set_xlabel('epoch'); a2.set_ylabel('validation R@1'); a2.set_ylim(0, 1); a2.set_title('Validation after each epoch')
plt.tight_layout(); plt.show()

## 7. The test: new domains the model has never seen

Now the fine-tuned model is scored on the test domains, alone and fused with TF-IDF (the way stage 1 combines signals).

In [ ]:
tuned = codebert_scorer(enc)
results['CodeBERT (fine-tuned)'] = evaluate(tuned, data['test'], keep_ranks=True)
results['TF-IDF + CodeBERT (fine-tuned)'] = evaluate(fused(tf, tuned), data['test'])
for pool in ('group pool', 'mixed pool'):
    print(pool, '(unseen domains)'); show_table({k: v[pool] for k, v in results.items()}, ['R@1', 'R@3', 'R@5', 'R@10', 'MRR']); print()

### Graph 2: all methods side by side

For each method: how often is the right code first (R@1), in the top 3, in the top 5. Left = the demanding group pool, right = the mixed pool.

In [ ]:
names = ['TF-IDF', 'MiniLM', 'CodeBERT (not trained)', 'CodeBERT (fine-tuned)', 'TF-IDF + CodeBERT (fine-tuned)']
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, pool in zip(axes, ('group pool', 'mixed pool')):
    w = 0.16
    for j, name in enumerate(names):
        vals = [results[name][pool][k] for k in ('R@1', 'R@3', 'R@5')]
        bars = ax.bar(np.arange(3) + (j - 2) * w, vals, w, color=color(name), label=name)
        for b_, v_ in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v_ + .01, f'{v_:.2f}', ha='center', fontsize=7)
    ax.set_xticks(range(3)); ax.set_xticklabels(['right code is first', 'in top 3', 'in top 5']); ax.set_ylim(0, 1.12); ax.set_title(pool)
axes[0].set_ylabel('share of requirements'); axes[0].legend(fontsize=7, loc='lower right')
plt.tight_layout(); plt.show()

### Graph 3: domain by domain

An average can hide a weak domain. Here is R@1 for each test domain, MiniLM against the fine-tuned model (group pool).

In [ ]:
mini_r = evaluate(mini, data['test'], keep_ranks=True)['_ranks']['per_group']; tuned_r = results['CodeBERT (fine-tuned)']['_ranks']['per_group']
gs = sorted(tuned_r); x = np.arange(len(gs)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(x - w / 2, [(mini_r[g] <= 1).mean() for g in gs], w, color=color('MiniLM'), label='MiniLM')
ax.bar(x + w / 2, [(tuned_r[g] <= 1).mean() for g in gs], w, color=color('CodeBERT (fine-tuned)'), label='CodeBERT (fine-tuned)')
ax.set_xticks(x); ax.set_xticklabels([g.split('/')[-1][:22] for g in gs], rotation=20, ha='right'); ax.set_ylabel('R@1'); ax.set_ylim(0, 1.05)
ax.set_title('R@1 per unseen test domain'); ax.legend(); plt.tight_layout(); plt.show()

### Graph 4: where does the right code land?

For every test requirement (mixed pool) we take the rank of its right code. A tall bar at rank 1 is good; a long tail to the right means the model often needs the right code to be found deeper in the list.

In [ ]:
cap = 15
fig, ax = plt.subplots(figsize=(8, 3.2))
for name, ranks in (('MiniLM', evaluate(mini, data['test'], keep_ranks=True)['_ranks']['mixed']), ('CodeBERT (fine-tuned)', results['CodeBERT (fine-tuned)']['_ranks']['mixed'])):
    shown = np.minimum(ranks, cap + 1); hist = np.bincount(shown, minlength=cap + 2)[1:] / len(ranks)
    ax.plot(range(1, cap + 2), hist, marker='o', color=color(name), label=name)
ax.set_xticks(range(1, cap + 2)); ax.set_xticklabels([str(i) for i in range(1, cap + 1)] + [f'>{cap}'])
ax.set_xlabel('rank of the right code'); ax.set_ylabel('share of requirements'); ax.set_title('Rank of the right code (mixed pool)'); ax.legend(); plt.tight_layout(); plt.show()

## 8. Control test: does the model really read the requirement?

A model could cheat by recognising a *style* of code (for example "code with validation logic") without reading the requirement. To check, every requirement is replaced by the requirement of its neighbour in the same domain and the scores are recomputed. If the model really matches meaning, R@1 must **collapse to about chance** (1 divided by the number of candidates). If it stayed high, the model would be using a shortcut.

In [ ]:
def shuffled_group_r1(score_fn, items):
    by_group, hits, chance = {}, [], []
    for it in items: by_group.setdefault(it['group'], []).append(it)
    for its in by_group.values():
        docs, right = pool_of(its); queries = [i['query'] for i in its]; rotated = queries[1:] + queries[:1]     # each requirement gets its neighbour's text
        ranks = rank_of_right(score_fn(rotated, docs), right); hits.append((ranks <= 1).mean()); chance.append(1 / len(docs))
    return float(np.mean(hits)), float(np.mean(chance))

real_r1 = results['CodeBERT (fine-tuned)']['group pool']['R@1']
shuf_r1, chance = shuffled_group_r1(tuned, data['test'])
print(f'real requirements    R@1 = {real_r1:.3f}\nshuffled requirements R@1 = {shuf_r1:.3f}\nchance level (1 / candidates) = {chance:.3f}')
fig, ax = plt.subplots(figsize=(5.5, 3)); ax.bar(['real\nrequirements', 'shuffled\nrequirements', 'chance'], [real_r1, shuf_r1, chance], color=['#d1495b', '#9aa5b1', '#cccccc'])
ax.set_ylabel('R@1 (group pool)'); ax.set_ylim(0, 1); ax.set_title('Control: the model must stop working when the requirement is wrong'); plt.tight_layout(); plt.show()
print('PASS: the model uses the requirement.' if shuf_r1 < 0.15 else 'WARNING: R@1 stayed high with shuffled requirements, the model may use a shortcut.')

## 9. Look at the mistakes

Numbers hide *how* a model is wrong. Here are three test requirements where the fine-tuned model put a wrong code first (group pool): the requirement, what the model chose, and the right code.

In [ ]:
shown = 0
by_group = {}
for it in data['test']: by_group.setdefault(it['group'], []).append(it)
for g, its in by_group.items():
    docs, right = pool_of(its); sc = tuned([i['query'] for i in its], docs)
    for i, r in enumerate(right):
        top = int(np.argmax(sc[i]))
        if top != r and shown < 3:
            print('REQUIREMENT:', its[i]['query']); print('MODEL CHOSE:\n', docs[top][:260].replace('\n', ' '), '...'); print('RIGHT CODE:\n', docs[r][:260].replace('\n', ' '), '...\n' + '-' * 60); shown += 1

## 10. Save the model and the results

The fine-tuned model is saved in a **new** folder (`models/c4-codebert-new`), so the model that the app uses now is not overwritten. A small file `c4_embedder.json` records the 256-token limit used in training; the app reads it. To switch the app to this model, copy the folder to `models/c4-codebert` (or set `EXTRA_EMBEDDING_MODEL_NAMES=c4-codebert-new` in `.env`). The result table is saved as JSON for the report. On Colab the model is also zipped for download.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
enc.model.save_pretrained(OUT); enc.tok.save_pretrained(OUT)
json.dump({'max_seq_length': 256, 'base': BASE}, open(OUT / 'c4_embedder.json', 'w'))
clean = {k: {p: v[p] for p in ('group pool', 'mixed pool')} for k, v in results.items()}
res_path = (Path('/content/out') if IN_COLAB else ROOT / 'data' / 'results') / 'codebert_training_notebook.json'
res_path.parent.mkdir(parents=True, exist_ok=True)
res_path.write_text(json.dumps({'config': {'base': BASE, 'epochs': EPOCHS, 'batch': BATCH, 'lr': LR, 'tau': TAU, 'quick': QUICK, 'test_groups': sorted(test_groups),
                                           'train_items': len(data['train']), 'test_items': len(data['test'])}, 'history': history, 'results': clean,
                                'control': {'real_R@1': real_r1, 'shuffled_R@1': shuf_r1, 'chance': chance}}, indent=1))
print('model saved in', OUT, '| results saved in', res_path)
if IN_COLAB:
    import shutil; shutil.make_archive('/content/c4-codebert-new', 'zip', '/content', 'out')
    from google.colab import files; files.download('/content/c4-codebert-new.zip')

## What to say about this notebook

* CodeBERT fine-tuned with a contrastive loss on requirement/code pairs finds the right code much more often than MiniLM **in the kind of data it was trained on**, in domains it had not seen.
* The control test shows it actually reads the requirement (it collapses when the requirement is shuffled).
* This is the *ceiling-style* test. Whether it helps on real projects is answered by the cross-validation notebook and the real-project notebook.